# Deep Learning Diagnostics Lab
CIFAR-10 small CNN, SGD vs AdamW. Drive에는 분석 결과만 저장하고 모델 weight는 `/content/local_checkpoints`에만 둡니다.

진단 시점: gradient/update=batch, loss/accuracy=epoch, representation=학습 전체의 약 4개 간격.

In [ ]:
!pip -q install umap-learn tensorboard
import json, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torch.utils.tensorboard import SummaryWriter
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import umap.umap_ as umap
from google.colab import drive

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED=7; FAST=True
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/deep_learning_diagnostics")
CSV,NPZ,FIG,TB,SUM=[ROOT/x for x in ["csv","npz","figures","tensorboard","summaries"]]
CKPT=Path("/content/local_checkpoints")
for p in [CSV,NPZ,FIG,TB,SUM,CKPT]: p.mkdir(parents=True,exist_ok=True)

norm=((.4914,.4822,.4465),(.2470,.2435,.2616))
train_tf=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(*norm)])
eval_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(*norm)])
aug=datasets.CIFAR10("/content/data",train=True,download=True,transform=train_tf)
evalset=datasets.CIFAR10("/content/data",train=True,download=False,transform=eval_tf)
perm=torch.randperm(len(aug),generator=torch.Generator().manual_seed(SEED)).tolist()
n_train,n_val,EPOCHS=(12000,2000,6) if FAST else (40000,5000,12)
tr_idx,va_idx=perm[:n_train],perm[n_train:n_train+n_val]
kw=dict(batch_size=256,num_workers=2,pin_memory=True)
train_loader=DataLoader(Subset(aug,tr_idx),shuffle=True,**kw)
train_eval_loader=DataLoader(Subset(evalset,tr_idx),shuffle=False,**kw)
val_loader=DataLoader(Subset(evalset,va_idx),shuffle=False,**kw)
interval=max(1,round(EPOCHS/4))
DIAG_EPOCHS=sorted(set([0]+list(range(interval,EPOCHS+1,interval))+[EPOCHS]))
print("device",DEVICE,"diagnostic epochs",DIAG_EPOCHS)

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv2d(3,32,3,padding=1),nn.ReLU())
        self.block1=nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.block2=nn.Sequential(nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.penultimate=nn.Linear(128,64)
        self.head=nn.Linear(64,10)
    def forward(self,x,return_features=False):
        f={}
        x=self.stem(x); f["stem"]=x.mean((2,3))
        x=self.block1(x); f["block1"]=x.mean((2,3))
        x=self.block2(x); f["block2"]=x.mean((2,3))
        x=self.pool(x).flatten(1)
        x=F.relu(self.penultimate(x)); f["penultimate"]=x
        logits=self.head(x)
        return (logits,f) if return_features else logits

LAYERS=["stem","block1","block2","penultimate"]
TRACK={"stem":"stem.0.weight","block1":"block1.0.weight","block2":"block2.0.weight","penultimate":"penultimate.weight","head":"head.weight"}
torch.manual_seed(SEED)
INIT={k:v.cpu().clone() for k,v in SmallCNN().state_dict().items()}

In [ ]:
@torch.no_grad()
def evaluate(model,loader):
    model.eval(); loss=correct=count=0
    for x,y in loader:
        x,y=x.to(DEVICE),y.to(DEVICE); logits=model(x)
        loss+=F.cross_entropy(logits,y,reduction="sum").item()
        correct+=(logits.argmax(1)==y).sum().item(); count+=len(y)
    return loss/count,correct/count

def train(run,optimizer_name):
    model=SmallCNN().to(DEVICE); model.load_state_dict(INIT)
    if optimizer_name=="sgd":
        optimizer=torch.optim.SGD(model.parameters(),lr=.08,momentum=.9)
    else:
        optimizer=torch.optim.AdamW(model.parameters(),lr=2e-3)
    writer=SummaryWriter(str(TB/run)); history=[]; dynamics=[]; step=0
    torch.save(model.state_dict(),CKPT/f"{run}_epoch0.pt")
    for epoch in range(1,EPOCHS+1):
        model.train(); total=correct=count=0
        for x,y in train_loader:
            x,y=x.to(DEVICE),y.to(DEVICE); optimizer.zero_grad(set_to_none=True)
            logits=model(x); loss=F.cross_entropy(logits,y); loss.backward()
            params=dict(model.named_parameters())
            before={tag:params[name].detach().clone() for tag,name in TRACK.items()}
            row={"run":run,"epoch":epoch,"step":step}
            for tag,name in TRACK.items(): row[tag+"_grad"]=params[name].grad.norm().item()
            optimizer.step(); params=dict(model.named_parameters())
            for tag,name in TRACK.items():
                delta=(params[name].detach()-before[tag]).norm().item()
                row[tag+"_update_weight"]=delta/(params[name].detach().norm().item()+1e-12)
                writer.add_scalar("grad/"+tag,row[tag+"_grad"],step)
                writer.add_scalar("update_weight/"+tag,row[tag+"_update_weight"],step)
            dynamics.append(row); step+=1
            total+=loss.item()*len(y); correct+=(logits.argmax(1)==y).sum().item(); count+=len(y)
        val_loss,val_acc=evaluate(model,val_loader)
        history.append({"run":run,"epoch":epoch,"train_loss":total/count,"train_acc":correct/count,"val_loss":val_loss,"val_acc":val_acc})
        writer.add_scalar("val/loss",val_loss,epoch); writer.add_scalar("val/acc",val_acc,epoch)
        if epoch in DIAG_EPOCHS: torch.save(model.state_dict(),CKPT/f"{run}_epoch{epoch}.pt")
        print(run,epoch,f"val_loss={val_loss:.3f}",f"val_acc={val_acc:.3f}")
    writer.close()
    history=pd.DataFrame(history); dynamics=pd.DataFrame(dynamics)
    history.to_csv(CSV/f"{run}_history.csv",index=False); dynamics.to_csv(CSV/f"{run}_dynamics.csv",index=False)
    return model,history,dynamics

sgd,sgd_hist,sgd_dyn=train("sgd","sgd")
adamw,adam_hist,adam_dyn=train("adamw","adamw")

%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/deep_learning_diagnostics/tensorboard

fig,ax=plt.subplots(1,2,figsize=(11,4))
for h,name in [(sgd_hist,"SGD"),(adam_hist,"AdamW")]:
    ax[0].plot(h.epoch,h.val_loss,"o-",label=name); ax[1].plot(h.epoch,h.val_acc,"o-",label=name)
for a in ax:a.legend()
ax[0].set_title("validation loss"); ax[1].set_title("validation accuracy")
plt.savefig(FIG/"training_curves.png",dpi=170); plt.show()

In [ ]:
@torch.no_grad()
def extract(model,loader,max_samples=2000):
    model.eval(); parts={k:[] for k in LAYERS}; labels=[]; preds=[]; count=0
    for x,y in loader:
        logits,features=model(x.to(DEVICE),True); take=min(len(y),max_samples-count)
        for layer in LAYERS: parts[layer].append(features[layer][:take].cpu())
        labels.append(y[:take]); preds.append(logits[:take].argmax(1).cpu()); count+=take
        if count>=max_samples: break
    return {k:torch.cat(v).numpy() for k,v in parts.items()},torch.cat(labels).numpy(),torch.cat(preds).numpy()

def checkpoint(run,epoch):
    return torch.load(CKPT/f"{run}_epoch{epoch}.pt",map_location="cpu")

def snapshots(run):
    result={}
    for epoch in DIAG_EPOCHS:
        model=SmallCNN().to(DEVICE); model.load_state_dict(checkpoint(run,epoch))
        f,y,p=extract(model,val_loader); result[epoch]=(f,y,p)
        np.savez_compressed(NPZ/f"{run}_epoch{epoch}.npz",labels=y,predictions=p,**f)
    return result

def erank(X):
    X=torch.tensor(X).float(); X-=X.mean(0); s=torch.linalg.svdvals(X); p=s.square(); p/=p.sum()
    return torch.exp(-(p*torch.log(p.clamp_min(1e-12))).sum()).item()

def cka(X,Y):
    X,Y=torch.tensor(X).float(),torch.tensor(Y).float(); X-=X.mean(0); Y-=Y.mean(0)
    return ((X.T@Y).square().sum()/(((X.T@X).square().sum().sqrt()*(Y.T@Y).square().sum().sqrt())+1e-12)).item()

S=snapshots("sgd"); A=snapshots("adamw"); rows=[]
for run,Z in [("sgd",S),("adamw",A)]:
    for epoch in DIAG_EPOCHS:
        for layer in LAYERS:
            rows.append({"run":run,"epoch":epoch,"layer":layer,"effective_rank":erank(Z[epoch][0][layer]),
                         "cka_init":cka(Z[epoch][0][layer],Z[0][0][layer]),"cka_final":cka(Z[epoch][0][layer],Z[EPOCHS][0][layer])})
rep=pd.DataFrame(rows); rep.to_csv(CSV/"representation_dynamics.csv",index=False)

for metric in ["effective_rank","cka_init","cka_final"]:
    fig,ax=plt.subplots(1,2,figsize=(11,4))
    for a,run in zip(ax,["sgd","adamw"]):
        for layer in LAYERS:
            q=rep[(rep.run==run)&(rep.layer==layer)]; a.plot(q.epoch,q[metric],"o-",label=layer)
        a.set_title(run+" "+metric); a.legend()
    plt.savefig(FIG/f"{metric}.png",dpi=170); plt.show()

for run,Z in [("sgd",S),("adamw",A)]:
    for epoch in DIAG_EPOCHS:
        X=StandardScaler().fit_transform(Z[epoch][0]["penultimate"])
        for name,reducer in [("pca",PCA(2)),("umap",umap.UMAP(n_components=2,n_neighbors=20,min_dist=.15,random_state=SEED))]:
            z=reducer.fit_transform(X); y,p=Z[epoch][1],Z[epoch][2]
            df=pd.DataFrame({"x":z[:,0],"y":z[:,1],"label":y,"pred":p}); df.to_csv(CSV/f"{run}_{name}_{epoch}.csv",index=False)
            plt.figure(figsize=(6,5)); plt.scatter(df.x,df.y,c=df.label,s=8,cmap="tab10"); wrong=df[df.label!=df.pred]
            plt.scatter(wrong.x,wrong.y,facecolors="none",edgecolors="black",s=30); plt.title(f"{run} {name} epoch {epoch}")
            plt.savefig(FIG/f"{run}_{name}_{epoch}.png",dpi=170); plt.show()

In [ ]:
final,y,p=S[EPOCHS]; X=StandardScaler().fit_transform(final["penultimate"])
neighbors=NearestNeighbors(n_neighbors=31).fit(X); _,idx=neighbors.kneighbors(X); local_dim=[]
for i in range(len(X)):
    local=X[idx[i,1:]]; local-=local.mean(0); s=np.linalg.svd(local,compute_uv=False); power=s*s
    local_dim.append(np.searchsorted(np.cumsum(power)/power.sum(),.9)+1)
pd.DataFrame({"label":y,"pred":p,"local_dim90":local_dim}).to_csv(CSV/"local_dimension.csv",index=False)

writer=SummaryWriter(str(TB/"projector_sgd")); metadata=[f"label={a} pred={b}" for a,b in zip(y,p)]
for layer in LAYERS: writer.add_embedding(torch.tensor(final[layer]).float(),metadata=metadata,tag=layer)
writer.close()

train_f,train_y,_=extract(sgd,train_eval_loader,5000)
def probe(Xtr,ytr,Xva,yva):
    scaler=StandardScaler(); Xtr=scaler.fit_transform(Xtr).astype("float32"); Xva=scaler.transform(Xva).astype("float32")
    Xtr,ytr=torch.tensor(Xtr,device=DEVICE),torch.tensor(ytr,device=DEVICE); Xva,yva=torch.tensor(Xva,device=DEVICE),torch.tensor(yva,device=DEVICE)
    head=nn.Linear(Xtr.shape[1],10).to(DEVICE); opt=torch.optim.AdamW(head.parameters(),lr=.08)
    for _ in range(150): opt.zero_grad(); F.cross_entropy(head(Xtr),ytr).backward(); opt.step()
    return (head(Xva).argmax(1)==yva).float().mean().item()
probe_df=pd.DataFrame([{"layer":l,"accuracy":probe(train_f[l],train_y,final[l],y)} for l in LAYERS]); probe_df.to_csv(CSV/"linear_probe.csv",index=False)

summary={"epochs":EPOCHS,"diagnostic_epochs":DIAG_EPOCHS,"effective_rank":{l:erank(final[l]) for l in LAYERS},
         "cka_sgd_adamw":{l:cka(final[l],A[EPOCHS][0][l]) for l in LAYERS},"local_dim90_mean":float(np.mean(local_dim)),
         "linear_probe":dict(zip(probe_df.layer,probe_df.accuracy.astype(float)))}
(SUM/"summary.json").write_text(json.dumps(summary,indent=2))
assert not any(ROOT.rglob("*.pt")),"Model weight found on Drive"
print(json.dumps(summary,indent=2))